<a href="https://colab.research.google.com/github/kavya-83175/24B11DS151_DriveReady_OPC_2026/blob/main/FinMate_Day_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain langchain-groq

In [2]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.3
)

print("FinMate LLM initialized successfully!")

FinMate LLM initialized successfully!


In [7]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

SYSTEM = """
ROLE:
You are FinMate AI, a friendly finance assistant
for small retail business owners.

FACTS:
You help users understand income, expenses,
sales, costs and financial records.

RULES:
Never invent income, expenses, balances or profit.
Ask for missing transaction details.
Never claim that transactions are permanently saved.
Use verified information for financial calculations.
If information is missing, clearly ask the user.

FORMAT:
Keep responses under 60 words.
Use simple and friendly English or Telugu according to user preference.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])

chain = prompt | llm | StrOutputParser()

history = []

def ask(question: str) -> str:
    answer = chain.invoke({
        "history": history,
        "question": question
    })

    history.extend([
        ("human", question),
        ("ai", answer)
    ])

    return answer

print(ask("Hi, I run a grocery shop."))
print(ask("What type of business do I run?"))

Hello! 👋 I’m happy to help with your grocery shop’s finances. What would you like to look at today—daily sales, expense tracking, profit calculation, or something else? Let me know any recent numbers you have, and we’ll get started!
You run a grocery shop—a small retail business that sells food and household items to local customers.


In [8]:
history

[('human', 'Hi, I run a grocery shop.'),
 ('ai',
  'Hello! 👋 I’m happy to help with your grocery shop’s finances. What would you like to look at today—daily sales, expense tracking, profit calculation, or something else? Let me know any recent numbers you have, and we’ll get started!'),
 ('human', 'What type of business do I run?'),
 ('ai',
  'You run a grocery shop—a small retail business that sells food and household items to local customers.')]

In [9]:
%%writefile /content/test.csv
question,must_include
What is income?,money
What is an expense?,expense
How can I calculate profit?,income
I earned Rs 8000 today. What is this?,income
I paid Rs 500 electricity bill. What is this?,expense
What information is needed to calculate profit?,expenses
Can you help me understand sales?,sales
How should I record stock purchases?,stock
Can you calculate monthly profit?,records
What financial information can you help with?,financial

Writing /content/test.csv


In [11]:
import csv

def run_tests(path: str = "/content/test.csv") -> None:

    with open(path, newline="", encoding="utf-8") as file:
        rows = list(csv.DictReader(file))

    passed = 0

    for row in rows:

        answer = chain.invoke({
            "history": [],
            "question": row["question"]
        })

        ok = row["must_include"].lower() in answer.lower()

        passed += ok

        print(
            "PASS" if ok else "FAIL",
            "|",
            row["question"],
            "->",
            answer[:70]
        )

    print(
        f"\n{passed}/{len(rows)} passed "
        f"({passed / len(rows):.0%})"
    )

In [12]:
run_tests()

PASS | What is income? -> Income is the money your business earns from selling goods or services
PASS | What is an expense? -> An expense is money you spend to run your business—like rent, utilitie
FAIL | How can I calculate profit? -> Sure! Profit = **Revenue (sales)** – **Expenses**.

- **Gross profit**
PASS | I earned Rs 8000 today. What is this? -> That’s a cash inflow – an **income** entry. If it’s from selling goods
PASS | I paid Rs 500 electricity bill. What is this? -> That’s an **expense** – a utility cost for electricity. To log it, cou
PASS | What information is needed to calculate profit? -> To find profit you need:

1. **Total revenue** (all sales/income).  
2
PASS | Can you help me understand sales? -> Absolutely! Sales are the total money you earn from selling products o
PASS | How should I record stock purchases? -> Record each purchase as a **Stock Purchase** entry:

1. Date & vendor 
FAIL | Can you calculate monthly profit? -> Sure! To work out your monthly profit I’l

In [13]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=SYSTEM
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "How can I calculate my business profit?"
        }
    ]
})

print(result["messages"][-1].content)

Sure! Profit = Total Revenue – Total Expenses.

1. Add up all sales/income for the period.  
2. Add up every cost: rent, utilities, stock, wages, taxes, etc.  
3. Subtract the expense total from the revenue total.

Need any help adding specific numbers? Let me know!
